

Assignment no 10. Write a better auto-complete algorithm using an N-gram model (similar models are used for
translation, determining the author of a text, and speech recognition)



**Install & Import**

In [1]:
!pip install -q nltk
import nltk
from collections import defaultdict, Counter
import math
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

**Load and Clean Text**

In [2]:
corpus = """
Natural language processing enables computers to understand human language.
Language models are used in autocomplete systems.
Autocomplete predicts the next word based on context.
N-gram models use previous words to estimate probabilities.
"""


**Tokenization**

In [4]:
nltk.download('punkt_tab')
tokens = nltk.word_tokenize(corpus.lower())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


**Generate N-grams**

In [5]:
def generate_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


In [6]:
unigrams = generate_ngrams(tokens, 1)
bigrams = generate_ngrams(tokens, 2)
trigrams = generate_ngrams(tokens, 3)


**Count N-grams**

In [7]:
uni_counts = Counter(unigrams)
bi_counts = Counter(bigrams)
tri_counts = Counter(trigrams)

vocab = set(tokens)
V = len(vocab)


**Add Laplace Smoothing**

In [8]:
def trigram_prob(w1, w2, w3):
    return (tri_counts[(w1, w2, w3)] + 1) / (bi_counts[(w1, w2)] + V)


**Backoff Strategy (Critical)**

In [9]:
def backoff_prob(context, word):
    if len(context) >= 2:
        w1, w2 = context[-2], context[-1]
        return trigram_prob(w1, w2, word)

    elif len(context) == 1:
        return (bi_counts[(context[-1], word)] + 1) / (uni_counts[(context[-1],)] + V)

    else:
        return (uni_counts[(word,)] + 1) / (sum(uni_counts.values()) + V)


**Autocomplete Function (Top-k Predictions)**

In [10]:
def autocomplete(text, k=5):
    tokens = nltk.word_tokenize(text.lower())
    probs = {}

    for word in vocab:
        probs[word] = backoff_prob(tokens, word)

    ranked = sorted(probs.items(), key=lambda x: x[1], reverse=True)
    return ranked[:k]


**Test the Model**

In [11]:
autocomplete("language models")


[('are', 0.06896551724137931),
 ('n-gram', 0.034482758620689655),
 ('models', 0.034482758620689655),
 ('systems', 0.034482758620689655),
 ('.', 0.034482758620689655)]